# Elliptic Dataset Investigation — COMP8851

This notebook computes all the metrics required for the dataset report:
node/edge/feature counts, fraud vs normal counts, fraud %, imbalance ratio, graph type, global and local heterophily, and split info.

**Before running:** update `DATA_DIR` below to point to the folder where you unzipped the Kaggle download (should contain `elliptic_txs_features.csv`, `elliptic_txs_edgelist.csv`, `elliptic_txs_classes.csv`).

In [ ]:
import pandas as pd
import numpy as np

# EDIT THIS PATH to wherever you unzipped the Kaggle download
DATA_DIR = "./elliptic_bitcoin_dataset"

features_path = f"{DATA_DIR}/elliptic_txs_features.csv"
edges_path = f"{DATA_DIR}/elliptic_txs_edgelist.csv"
classes_path = f"{DATA_DIR}/elliptic_txs_classes.csv"

print("Paths set. Make sure these three files exist at:")
print(features_path)
print(edges_path)
print(classes_path)

## Step 1: Load the data

The features file has no header (txId, time_step, 165 features). The classes file has a header (txId, class) where class is '1' (illicit), '2' (licit), or 'unknown'. The edgelist has columns txId1, txId2.

In [ ]:
features = pd.read_csv(features_path, header=None)
classes = pd.read_csv(classes_path)
edges = pd.read_csv(edges_path)

# Name the first two feature columns
features = features.rename(columns={0: "txId", 1: "time_step"})

print("Features shape:", features.shape)
print("Classes shape:", classes.shape)
print("Edges shape:", edges.shape)
print()
print(classes.head())

## Step 2: Basic counts (nodes, edges, features)

In [ ]:
num_nodes = features.shape[0]
num_edges = edges.shape[0]
num_features = features.shape[1] - 2  # minus txId and time_step columns

print(f"Number of nodes: {num_nodes}")
print(f"Number of edges: {num_edges}")
print(f"Number of features per node: {num_features}")

## Step 3: Label / fraud statistics

Class '1' = illicit (fraud), '2' = licit (normal), 'unknown' = unlabeled.

In [ ]:
classes["class"] = classes["class"].astype(str)

fraud_nodes = (classes["class"] == "1").sum()
normal_nodes = (classes["class"] == "2").sum()
unknown_nodes = (classes["class"] == "unknown").sum()

labeled_total = fraud_nodes + normal_nodes
fraud_pct_of_labeled = fraud_nodes / labeled_total * 100
fraud_pct_of_all = fraud_nodes / num_nodes * 100
imbalance_ratio = normal_nodes / fraud_nodes

print(f"Fraud (illicit) nodes: {fraud_nodes}")
print(f"Normal (licit) nodes: {normal_nodes}")
print(f"Unknown/unlabeled nodes: {unknown_nodes}")
print(f"Total labeled nodes: {labeled_total}")
print(f"Fraud % (of labeled nodes): {fraud_pct_of_labeled:.2f}%")
print(f"Fraud % (of ALL nodes incl. unknown): {fraud_pct_of_all:.2f}%")
print(f"Imbalance ratio (normal:fraud): {imbalance_ratio:.2f} : 1")

## Step 4: Graph type checks

Elliptic is homogeneous (one node type: transactions; one relation type: payment flow) and dynamic (spans 49 time steps).

In [ ]:
num_time_steps = features["time_step"].nunique()

print("Graph type: Homogeneous (single node type: transactions)")
print("Node types: 1")
print("Relation types: 1 (payment flow / transaction edge)")
print(f"Static or dynamic: Dynamic — spans {num_time_steps} time steps")
print("Fraud type: Organic/real (investigator-confirmed illicit transactions, not injected)")

## Step 5: Global heterophily ratio

Global heterophily = (edges connecting two DIFFERENT-label nodes) / (total edges where BOTH endpoints are labeled).
Only edges between two labeled nodes count — edges touching an 'unknown' node are excluded.

In [ ]:
# Build a fast lookup: txId -> label ('1', '2', or 'unknown')
label_map = dict(zip(classes["txId"], classes["class"]))

edges.columns = ["txId1", "txId2"]

labels1 = edges["txId1"].map(label_map)
labels2 = edges["txId2"].map(label_map)

# Keep only edges where BOTH endpoints are labeled (1 or 2, not unknown/missing)
both_labeled_mask = labels1.isin(["1", "2"]) & labels2.isin(["1", "2"])
labeled_edges = edges[both_labeled_mask].copy()
l1 = labels1[both_labeled_mask]
l2 = labels2[both_labeled_mask]

total_labelled_edges = len(labeled_edges)
different_label_edges = (l1 != l2).sum()

global_heterophily = different_label_edges / total_labelled_edges

print(f"Total edges with both endpoints labeled: {total_labelled_edges}")
print(f"Edges connecting different-label nodes: {different_label_edges}")
print(f"Global heterophily ratio: {global_heterophily:.4f}")

## Step 6: Local heterophily per node (mean, median, std)

For each labeled node, local heterophily = (fraction of its labeled neighbors with a DIFFERENT label from itself).
Nodes with zero labeled neighbors are skipped (undefined local heterophily).

In [ ]:
from collections import defaultdict

# Build an undirected adjacency list using only labeled edges
adj = defaultdict(set)
for a, b in zip(labeled_edges["txId1"], labeled_edges["txId2"]):
    adj[a].add(b)
    adj[b].add(a)

local_heterophily = []

for node, neighbors in adj.items():
    node_label = label_map.get(node)
    if node_label not in ("1", "2"):
        continue
    if len(neighbors) == 0:
        continue
    diff_count = sum(1 for n in neighbors if label_map.get(n) != node_label)
    local_heterophily.append(diff_count / len(neighbors))

local_heterophily = np.array(local_heterophily)

print(f"Number of labeled nodes with at least 1 labeled neighbor: {len(local_heterophily)}")
print(f"Local heterophily — mean:   {local_heterophily.mean():.4f}")
print(f"Local heterophily — median: {np.median(local_heterophily):.4f}")
print(f"Local heterophily — std:    {local_heterophily.std():.4f}")

## Step 7: Split info (reference — the original paper uses a temporal split)

The original Elliptic paper (Weber et al., 2019) splits by time step: the first ~34 time steps are typically used for training, and the last ~15 for testing. There is no single official validation split in the raw release — teams commonly hold out a slice of the training time steps for validation. Adjust these numbers if your project defines its own split.

In [ ]:
print("Original split type: Temporal (NOT random)")
print("Common convention: time steps 1-34 = train, time steps 35-49 = test")
print("No official validation split provided in the raw Kaggle release —")
print("most papers carve a validation slice out of the training time steps themselves.")

## Step 8: Summary table

Run this last to get one clean block you can copy into your report / shared table.

In [ ]:
summary = {
    "Nodes": num_nodes,
    "Edges": num_edges,
    "Features": num_features,
    "Fraud nodes": fraud_nodes,
    "Normal nodes": normal_nodes,
    "Unknown/unlabeled nodes": unknown_nodes,
    "Fraud % (of labeled)": round(fraud_pct_of_labeled, 2),
    "Imbalance ratio (normal:fraud)": round(imbalance_ratio, 2),
    "Homogeneous/Heterogeneous": "Homogeneous",
    "Node types": 1,
    "Relation types": 1,
    "Static/Dynamic": f"Dynamic ({num_time_steps} time steps)",
    "Organic/Synthetic fraud": "Organic",
    "Global heterophily ratio": round(global_heterophily, 4),
    "Local heterophily - mean": round(local_heterophily.mean(), 4),
    "Local heterophily - median": round(float(np.median(local_heterophily)), 4),
    "Local heterophily - std": round(local_heterophily.std(), 4),
    "Original split": "Temporal (time steps 1-34 train / 35-49 test, no official val split)",
}

for k, v in summary.items():
    print(f"{k}: {v}")

## Notes to fill in manually (not computable from the CSVs)

- **Dataset/paper link:** https://www.kaggle.com/datasets/ellipticco/elliptic-data-set (paper: Weber et al., 2019, "Anti-Money Laundering in Bitcoin")
- **GitHub/download link:** Kaggle link above; official repo reference in the paper
- **Limitations/compatibility issues:** ~78% of nodes are unlabeled (only ~46K of ~204K are labeled), so most fraud/imbalance statistics above are computed over the labeled subset only. No official validation split is provided. License requires checking Kaggle's terms before public redistribution.